# 05 Incremental Federated Learning

Runs the resumable all-client, trust-weighted federated baseline initialized from the stage-04 encoder.

In [4]:
from pathlib import Path
import os
import sys
import shutil
import subprocess

ABS_PROJECT_ROOT = Path('/Volumes/LocalDrive1/onlyscholar-projects/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
EXPECTED_DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
DRIVE_PROJECT_ROOT = EXPECTED_DRIVE_PROJECT_ROOT

# Validate the project itself, not only the mount directory: Colab can leave a stale /content/drive/MyDrive folder.
def is_project_root(path):
    return (path / 'configs' / 'project_colab.json').is_file() and (path / 'src' / 'ecotrust').is_dir()


def discover_mounted_project():
    candidates = [EXPECTED_DRIVE_PROJECT_ROOT]
    my_drive = Path('/content/drive/MyDrive')
    candidates.extend([
        my_drive / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
        my_drive / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
    ])
    shared_drives = Path('/content/drive/Shareddrives')
    if shared_drives.is_dir():
        for shared_drive in shared_drives.iterdir():
            candidates.extend([
                shared_drive / 'projects' / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
                shared_drive / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
                shared_drive / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
            ])
    return next((candidate for candidate in candidates if is_project_root(candidate)), None)

try:
    from google.colab import drive  # type: ignore
    if not is_project_root(EXPECTED_DRIVE_PROJECT_ROOT):
        print('Drive project is not visible; resetting stale DriveFS state...', flush=True)
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        subprocess.run(['fusermount', '-uz', '/content/drive'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        shutil.rmtree('/content/drive', ignore_errors=True)
        Path('/content/drive').mkdir(parents=True, exist_ok=True)
        drive.mount('/content/drive', force_remount=False, timeout_ms=120000)
    discovered_root = discover_mounted_project()
    if discovered_root is not None:
        DRIVE_PROJECT_ROOT = discovered_root
except ImportError:
    pass
except Exception as exc:
    raise RuntimeError(f'Google Drive project mount failed: {type(exc).__name__}: {exc}') from exc


def find_project_root():
    candidates = []
    env_root = os.environ.get('ECOTRUST_PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path.cwd().parent, DRIVE_PROJECT_ROOT, ABS_PROJECT_ROOT])
    for base in list(candidates):
        candidates.extend(base.parents)
    seen = set()
    for candidate in candidates:
        expanded = candidate.expanduser()
        resolved = expanded.resolve() if expanded.exists() else expanded
        if str(resolved) in seen:
            continue
        seen.add(str(resolved))
        if (resolved / 'configs' / 'project.json').exists() and (resolved / 'src' / 'ecotrust').exists():
            return resolved
    raise RuntimeError(
        'Cannot find EcoTrust-FLIDS IMPLEMENTATION in the mounted Google Drive. Expected path: '
        + str(EXPECTED_DRIVE_PROJECT_ROOT) + '. Authorize the Google account that can open '
        'https://drive.google.com/drive/folders/1lWrVkEURlFciIae0chK8hjDUpVafGWkS, '
        'or add that folder as a shortcut under MyDrive/projects/implementation.'
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if str(PROJECT_ROOT).startswith('/content/drive') and (PROJECT_ROOT / 'configs' / 'project_colab.json').exists():
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project_colab.json'
else:
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project.json'
RUN_MODE = 'full'
DATASETS = ['Edge-IIoTset', 'TON_IoT', 'CICIoT2023']
print('PROJECT_ROOT:', PROJECT_ROOT)
print('CONFIG_PATH:', CONFIG_PATH)

from datetime import datetime, timezone
import importlib
import json
import traceback
import torch
from ecotrust.config import load_config, run_mode_settings
from ecotrust.data.clients import ClientSplit
from ecotrust.data.preprocess import load_preprocessed
from ecotrust.io import read_json, write_json
import ecotrust.federated.client as federated_client_module
import ecotrust.federated.server as federated_server_module
import ecotrust.models.embedding as embedding_module
import ecotrust.federated.incremental as federated_incremental_module
importlib.invalidate_caches()
for module in [federated_client_module, federated_server_module, embedding_module, federated_incremental_module]:
    importlib.reload(module)
load_embedding_model = embedding_module.load_embedding_model
run_incremental_federated_training = federated_incremental_module.run_incremental_federated_training
cfg = load_config(CONFIG_PATH)
settings = run_mode_settings(cfg, RUN_MODE)
DEVICE = 'cpu'
BATCH_SIZE = 512 if DEVICE == 'cuda' else 256
RUN_LOG = cfg.logs_root / f'05_incremental_federated_learning_{RUN_MODE}.log'
RUN_MANIFEST = cfg.logs_root / f'05_incremental_federated_learning_{RUN_MODE}_manifest.json'
cfg.logs_root.mkdir(parents=True, exist_ok=True)
manifest = {'stage': 5, 'mode': RUN_MODE, 'device': DEVICE, 'started_at': datetime.now(timezone.utc).isoformat(), 'datasets': {}}
write_json(RUN_MANIFEST, manifest)
print('DEVICE:', DEVICE)
print('EMBEDDING_SOURCE:', embedding_module.__file__)
print('FEDERATED_SOURCE:', federated_incremental_module.__file__)
print('RUN_LOG:', RUN_LOG)
for dataset in DATASETS:
    print('Federated', dataset, flush=True)
    prep = cfg.results_root / dataset / 'preprocessed' / RUN_MODE
    embedding_checkpoint = cfg.results_root / dataset / 'embedding' / RUN_MODE / 'ecotrust_encoder.pt'
    out = cfg.results_root / dataset / 'federated' / RUN_MODE
    progress_path = out / 'federated_progress.json'
    if progress_path.exists():
        progress = read_json(progress_path)
        if progress.get('status') == 'completed' and progress.get('completed_round') == settings['rounds']:
            print('Already completed; skipping', dataset, flush=True)
            manifest['datasets'][dataset] = {'status': 'completed', 'skipped': True}
            write_json(RUN_MANIFEST, manifest)
            continue
    X_train, X_test, y_train, y_test = load_preprocessed(prep)
    splits = [ClientSplit(**x) for x in json.loads((prep / 'client_splits.json').read_text())]
    model, checkpoint = load_embedding_model(embedding_checkpoint, DEVICE)
    if int(checkpoint['config']['input_dim']) != X_train.shape[1]:
        raise ValueError(f'{dataset}: encoder input dimension does not match preprocessed features')
    manifest['datasets'][dataset] = {'status': 'running', 'output_dir': str(out), 'embedding_checkpoint': str(embedding_checkpoint)}
    write_json(RUN_MANIFEST, manifest)
    try:
        result = run_incremental_federated_training(
            model, X_train, y_train, X_test, y_test, splits, out,
            rounds=settings['rounds'], local_epochs=settings['local_epochs'],
            batch_size=BATCH_SIZE, device=DEVICE, seed=cfg.seed, resume=True, log_path=RUN_LOG,
            selection_policy='all',
        )
        manifest['datasets'][dataset] = {'status': 'completed', 'final_metrics': result['final_metrics'], 'output_dir': str(out)}
        write_json(RUN_MANIFEST, manifest)
        print(dataset, result['final_metrics'], flush=True)
    except BaseException as exc:
        manifest['datasets'][dataset] = {'status': 'interrupted' if isinstance(exc, KeyboardInterrupt) else 'failed', 'error': f'{type(exc).__name__}: {exc}', 'traceback': traceback.format_exc(), 'output_dir': str(out)}
        write_json(RUN_MANIFEST, manifest)
        raise
manifest['completed_at'] = datetime.now(timezone.utc).isoformat()
manifest['status'] = 'completed'
write_json(RUN_MANIFEST, manifest)
print('Stage 05 complete:', RUN_MANIFEST)

PROJECT_ROOT: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION
CONFIG_PATH: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/configs/project_colab.json
DEVICE: cpu
EMBEDDING_SOURCE: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/src/ecotrust/models/embedding.py
FEDERATED_SOURCE: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/src/ecotrust/federated/incremental.py
RUN_LOG: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/logs/05_incremental_federated_learning_full.log
Federated Edge-IIoTset
Already completed; skipping Edge-IIoTset
Federated TON_IoT
[2026-10-23T07:31:20+00:00] Resuming from completed round 21/30 on cpu.
[2026-10-23T07:31:44+00:00] Round 22/30 complete: f1=0.963553, selected=20/20, energy_saving=0.00%.
[2026-10-23T07:32:09+00:00] Round 23/30 complete: f1=0.969990, selected=20/2